### Mercari PyCaret 분석기 클래스
1. TSV 데이터 로딩 지원
2. PyCaret setup, compare_models로 base model 탐색
3. 차원 축소(TF-IDF/Embedding 과 관련한 고차원 feature) 적용 가능
4. 단계별 진행 print 문구
5. tqdm 진행 표시
6. 모델 성능 지표 .json 저장
7. Submission CSV 저장
8. plot_model 시각화 저장 (../images/{model_name}_{timestamp}.png)

In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

from pycaret.regression import *

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD

from kjh_mercari_analyzer import MercariPyCaretAnalyzer

In [2]:
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file='train.tsv', test_file='test.tsv')
# analyzer.vectorize_text(method='tfidf', max_features=50000, n_components=100)
# analyzer.setup_pycaret()
# analyzer.find_base_model(sort_metric='R2')
# analyzer.save_metrics()
# analyzer.visualize_model(plots=['residuals','feature'])
# analyzer.predict_test(submission_file='submission.csv')

In [3]:
analyzer = MercariPyCaretAnalyzer(
    data_dir="../data", images_dir="../images", results_dir="../results"
)

In [4]:
analyzer.load_data()

📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
Price NaN count: 0
Final Price NaN count: 0
Train length: 1481661

Train head:
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name     price  \
0                                  Men/Tops/T-shirts    Unknown  2.397895   
1  Electronics/Computers & Tablets/Components & P...      Razer  3.970292   
2                        Women/Tops & Blouses/Blouse     Target  2.397895   
3                 Home/Home Décor/Home Décor Accents    Unknown  3.583519   
4 

In [5]:
analyzer.apply_undersampling(method="stratified", target_size=100000, n_bins=10)


🎯 Undersampling 시작...
   방법: stratified
   원본 크기: 1,481,661
   목표 크기: 100,000
   샘플링 비율: 6.75%
✅ Undersampling 완료
   최종 크기: 740,828
   실제 샘플링 비율: 50.00%
   제거된 샘플: 740,833

📊 샘플링 후 가격 분포:
   평균: 2.9809
   중앙값: 2.8904
   표준편차: 0.7462
   최소: 1.3863
   최대: 7.6034


In [6]:
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name


Text columns:  50%|█████     | 1/2 [00:46<00:46, 46.61s/it]

   ▪ 차원 축소 완료: 100 components
▶ 컬럼: item_description


Text columns: 100%|██████████| 2/2 [02:32<00:00, 76.29s/it]

   ▪ 차원 축소 완료: 100 components


✅ 벡터화 + 차원 축소 + 카테고리 인코딩 완료: train (740828, 205), test (693359, 205)


In [7]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(740828, 206)"
4,Transformed data shape,"(740828, 216)"
5,Transformed train set shape,"(518579, 216)"
6,Transformed test set shape,"(222249, 216)"
7,Numeric features,200
8,Categorical features,5
9,Preprocess,True


✅ PyCaret setup 완료


In [ ]:
analyzer.find_base_model(sort_metric="R2")

🔍 Base model 탐색 시작...


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,12:08:45
Status,. . . . . . . . . . . . . . . . . .,Fitting 10 Folds
Estimator,. . . . . . . . . . . . . . . . . .,K Neighbors Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lr,Linear Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,10.2940
ridge,Ridge Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,3.8410
lar,Least Angle Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,3.8710
br,Bayesian Ridge,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,14.3300
huber,Huber Regressor,0.4875,0.4132,0.6428,0.2579,0.1571,0.1697,10.0590
omp,Orthogonal Matching Pursuit,0.5249,0.4587,0.6772,0.1762,0.1667,0.1863,3.9370
lasso,Lasso Regression,0.5814,0.5567,0.7462,-0.0000,0.1849,0.2096,4.1710
en,Elastic Net,0.5814,0.5567,0.7462,-0.0000,0.1849,0.2096,3.9790
llar,Lasso Least Angle Regression,0.5814,0.5567,0.7462,-0.0000,0.1849,0.2096,3.9020
par,Passive Aggressive Regressor,0.7700,0.9683,0.9837,-0.7392,0.2543,0.2721,4.1410


Processing:   0%|          | 0/85 [00:00<?, ?it/s]

In [ ]:
analyzer.save_metrics()

In [ ]:
analyzer.visualize_model(plots=["residuals", "feature"])

In [ ]:
analyzer.predict_test(submission_file="submission.csv")

In [ ]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


TypeError: setup() got an unexpected keyword argument 'silent'

In [ ]:
analyzer.find_base_model(sort_metric="R2")
analyzer.save_metrics()  # JSON 저장
analyzer.visualize_model(plots=["residuals", "feature"])
analyzer.predict_test(submission_file="submission.csv")

In [ ]:
analyzer.vectorize_text()

In [ ]:
analyzer.setup_pycaret()
analyzer.find_base_model(sort_metric="R2")
analyzer.save_plot_model(plot_type="residuals")
analyzer.save_plot_model(plot_type="feature")
analyzer.predict_test()

# 1st try에서 차원축소를 하지 않아 메모리 에러남!!!
# MemoryError: Unable to allocate 671. GiB for an array with shape (1037162, 86798) and data type float64

In [ ]:
analyzer.find_base_model(sort_metric="R2")
analyzer.visualize_model(plots=["residuals", "feature_importance"])
submission = analyzer.predict_test(submission_file="../results/submission.csv")